# Llama-2 English-to-Bengali Translation Pipeline

This notebook implements an end-to-end pipeline for fine-tuning a Large Language Model (TinyLlama/Llama-2) for English-to-Bengali translation using the OPUS-100/BanglaNMT dataset.

The pipeline follows these key phases:
1.  **Environment Setup**: Installing dependencies and verifying hardware.
2.  **Configuration**: Centralizing model names and hyperparameters.
3.  **Data Acquisition**: Loading the English-Bengali translation dataset.
4.  **Preprocessing**: Tokenization and data formatting.
5.  **Model Preparation**: Quantization (4-bit) and LoRA configuration.
6.  **Training**: Fine-tuning using SFTTrainer.
7.  **Inference**: Testing the model on English-to-Bengali translation tasks.

## Phase 1: Environment Setup

Installing the necessary libraries for fine-tuning: `transformers`, `trl`, `peft`, `accelerate`, and `bitsandbytes`.

In [ ]:
!pip install -q transformers trl datasets accelerate peft bitsandbytes

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

### Library Imports

In [ ]:
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import load_dataset
from trl import SFTTrainer
import os
import torch

## Phase 2: Pipeline Configuration

Centralizing model names and training hyperparameters.

In [ ]:
# Model configuration
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
dataset_name = "Helsinki-NLP/opus-100"
dataset_config = "bn-en" 
new_model_name = "tinyllama-en-bn-translator"

# Training hyperparameters
output_dir = "./results_bn"
num_train_epochs = 1
max_steps = 1000
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
learning_rate = 2e-4
max_seq_length = 256

## Phase 3: Data Acquisition

Loading the English-to-Bengali translation dataset.

In [ ]:
print(f"Loading dataset: {dataset_name} ({dataset_config})")
dataset = load_dataset(dataset_name, dataset_config)

# Sample display
print("Sample data point:")
sample = dataset['train'][0]['translation']
print(f"EN: {sample['en']} | BN: {sample['bn']}")

## Phase 4: Data Preprocessing

Formatting the data for supervised fine-tuning.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True, add_eos_token=True)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = "right"

def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example['translation'])):
        en_text = example['translation'][i]['en']
        bn_text = example['translation'][i]['bn']
        text = f"English: {en_text} ###> Bengali: {bn_text}"
        output_texts.append(text)
    return output_texts

print("Tokenizer and formatting function prepared.")

## Phase 5: Model Preparation

Configuring 4-bit quantization and LoRA.

In [ ]:
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_name, 
    quantization_config=bnb_config, 
    device_map={ "": 0 }
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

## Phase 6: Training (Fine-Tuning)

Initializing the `SFTTrainer` and starting the fine-tuning process.

In [ ]:
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim="paged_adamw_32bit",
    save_steps=250,
    logging_steps=50,
    learning_rate=learning_rate,
    weight_decay=0.001,
    fp16=True,
    max_steps=max_steps,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    peft_config=peft_config,
    formatting_func=formatting_prompts_func,
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments
)

print("Training initialized. Run trainer.train() to start.")

## Phase 7: Inference & Translation Test

Testing the fine-tuned model.

In [ ]:
def translate_bn(text, model, tokenizer):
    prompt = f"English: {text} ###> Bengali:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=100, 
        do_sample=True, 
        top_k=50, 
        top_p=0.95
    )
    
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.split("###>")[1].replace("Bengali:", "").strip() if "###>" in result else result

# Test sentence
test_sentence = "Sustainability is key to our future."
print(f"English: {test_sentence}")
# print(f"Bengali: {translate_bn(test_sentence, model, tokenizer)}") # Uncomment after training